# DataPulse — Architecture Impact Analysis

_2026-05-13_

A one-page walkthrough of what the recent refactor delivered: a deterministic synthetic sales pipeline, a direct CSV → Neo4j knowledge graph, and a single google-adk Gemini agent that answers natural-language questions via read-only Cypher.

**To execute live cells:** `uv add --dev jupyter`, then `uv run jupyter lab notebooks/`. Cells that require a live Neo4j Aura instance or `GOOGLE_API_KEY` skip gracefully when those aren't configured.

## 1. Before vs. after

| | Before (last week) | After (today) |
|---|---|---|
| Graph layer | `NetworkXStore` stub (no real ops) | `Neo4jStore` + Aura, real persistence |
| Builder | `GraphBuilder.build()` raising `NotImplementedError` | `Neo4jGraphBuilder.build_from_csv` (UNWIND-batched MERGE) |
| Query path | Two stubs: `RuleBasedResolver` + `LLMResolver` (latter didn't actually read the graph) | One `google-adk` `Agent` with `run_cypher` tool |
| Read-only safety | none | regex-based guard rejects writes before they hit the driver |
| Dataset | 8-row `sample_sales.csv` | 1000-row deterministic CSV + quality report |
| LOC removed | — | 4 files / ~120 LOC |
| Test surface | 7 (domain + datagen scaffold) | 94 green (49 datagen + 16 graph + 29 query_engine + 4 domain) |

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CSV_PATH = RAW_DIR / "sales_1k.csv"
REPORT_PATH = RAW_DIR / "quality_report.json"

print(f"project_root:   {PROJECT_ROOT}")
print(f"csv present:    {CSV_PATH.exists()}")
print(f"report present: {REPORT_PATH.exists()}")

## 2. Synthetic sales data

1000 orders generated by `src/datagen/generate.py`. Same seed → byte-identical output. Run `uv run python -m src.datagen.generate` first if `data/raw/sales_1k.csv` is missing.

In [ ]:
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    print(f"shape: {df.shape}")
    display(df.head())
else:
    print("CSV not found. Generate it with: uv run python -m src.datagen.generate")

In [ ]:
if CSV_PATH.exists():
    for col in ("region", "channel", "category"):
        counts = df[col].value_counts()
        share = (counts / counts.sum() * 100).round(1)
        print(f"\n{col}:")
        for label, n in counts.items():
            print(f"  {label:<15}  {n:4d}  ({share[label]:>4}%)")

### Quality report — schema conformance + distributional realism

χ² goodness-of-fit on region / channel / category; per-row band checks on price and quantity; strict date-range check.

In [ ]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    s = report["summary"]
    print(f"pass: {s['pass']}   schema: {s['schema_pass']}   distribution: {s['distribution_pass']}   rows: {s['row_count']}\n")
    for name, d in report["distributions"].items():
        if d.get("chi2") is not None:
            print(f"  {name:<12}  chi2={d['chi2']:6.3f}  p={d['p_value']:.3f}  pass={d['passed']}")
        elif "violation_rate" in d:
            print(f"  {name:<12}  violation_rate={d['violation_rate']:.4f}  pass={d['passed']}")
        elif "out_of_range_rate" in d:
            print(f"  {name:<12}  out_of_range_rate={d['out_of_range_rate']:.4f}  pass={d['passed']}")
        else:
            print(f"  {name:<12}  pass={d['passed']}")
else:
    print("No quality report yet. Run datagen first.")

## 3. Graph schema

Six node labels (CamelCase, canonical Cypher strings) and five relationship types (UPPER_SNAKE_CASE). Defined once in `src/graph/domain/schema.py` and reused by the builder, the agent's schema card, and the tests — no duplication.

In [ ]:
from src.graph.domain.schema import EdgeType, NodeType

print("Node labels:")
for n in NodeType:
    print(f"  ({n.value})")
print("\nRelationships:")
for e in EdgeType:
    print(f"  -[:{e.value}]->")

### Idempotent CSV → graph load

One UNWIND-batched query per ~500 rows. All `MERGE`s — no `CREATE` — so re-running on the same CSV is a no-op after the first load.

In [ ]:
from src.graph.builder.neo4j_graph_builder import _MERGE_ORDER_ROW
print(_MERGE_ORDER_ROW)

## 4. Agent system prompt — generated from enums

The schema card the agent sees stays in sync with the code: change the enums in `schema.py`, the agent's `instruction` updates on the next process start.

In [ ]:
from src.query_engine.agent.schema_card import SCHEMA_CARD
print(SCHEMA_CARD)

## 5. Read-only guard — the agent can attempt anything; writes never reach Neo4j

The `run_cypher` tool wraps the driver. Before the query reaches the database, a word-bounded regex (comment-aware) rejects any write keyword. The agent receives a structured `error` and can reformulate.

In [ ]:
from src.query_engine.agent.cypher_tool import run_cypher

class _NoopStore:
    def run_read(self, q, **p):
        return [{"would_have_run": q}]

attempts = [
    "MATCH (c:Customer) RETURN c LIMIT 3",
    "CREATE (c:Customer {customer_id: 'X'}) RETURN c",
    "MATCH (n) DETACH DELETE n",
    "CALL apoc.create.node(['Customer'], {})",
    "// CREATE in comment is fine\nMATCH (n) RETURN n LIMIT 1",
]
for q in attempts:
    out = run_cypher(q, _NoopStore())
    head = q.split("\n")[0][:55]
    if "error" in out:
        print(f"REJECTED  {head:<55}  ->  {out['error'][:55]}")
    else:
        print(f"ACCEPTED  {head:<55}  ->  {out['row_count']} row(s)")

## 6. Live Q&A — the agent answering against Aura

Gated on `.env`. If `NEO4J_URI` and `GOOGLE_API_KEY` are populated and the CSV has been loaded via `uv run python -m src.graph.builder.neo4j_graph_builder --csv data/raw/sales_1k.csv`, the agent runs three sample questions below. Otherwise the cell prints the setup steps and exits.

In [ ]:
from src.shared.config import Settings

settings = Settings()
has_neo4j = bool(settings.neo4j_uri)
has_gemini = bool(settings.google_api_key)
print(f"NEO4J_URI set:    {has_neo4j}")
print(f"GOOGLE_API_KEY set: {has_gemini}")

if has_neo4j and has_gemini:
    from src.graph.store.neo4j_store import Neo4jStore
    from src.query_engine.agent.adk_agent import ask_async, build_agent

    questions = [
        "How many orders were placed in 2024?",
        "What are the top 3 product categories by total quantity?",
        "Which region has the highest average unit price?",
    ]
    with Neo4jStore.from_settings(settings) as store:
        agent = build_agent(store)
        for q in questions:
            print(f"\nQ: {q}")
            answer = await ask_async(agent, q)
            print(f"A: {answer}")
else:
    print("\nTo run live:")
    print("  1. Copy .env.sample to .env and fill in NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, GOOGLE_API_KEY")
    print("  2. uv run python -m src.datagen.generate")
    print("  3. uv run python -m src.graph.builder.neo4j_graph_builder --csv data/raw/sales_1k.csv")
    print("  4. Re-run this cell")

## 7. Test surface

94 tests green offline — the entire pipeline is covered with mocked Neo4j driver + mocked agent runner, so CI doesn't need Aura access.

In [ ]:
test_files = {
    "tests/test_sales_data.py":      "sales_data domain models (Order/Product/Customer)",
    "tests/test_graph.py":           "schema enums (CamelCase + UPPER_SNAKE values)",
    "tests/test_query_engine.py":    "Query / QueryResult dataclasses",
    "tests/datagen/":                "config / schema / generator / writer / reports / validator (42 tests)",
    "tests/graph/":                  "Neo4jStore + Neo4jGraphBuilder, mocked driver (16 tests)",
    "tests/query_engine/":           "cypher_tool + schema_card + adk_agent, mocked Runner (29 tests)",
}
for path, desc in test_files.items():
    print(f"  {path:<32} {desc}")
print("\nRun all:  uv run pytest tests/ -q")

## 8. What this architecture unlocks next

- **New question types** land by improving the agent's Cypher, not by adding code paths.
- **Schema changes** ripple through automatically: the schema card, the builder, the constraints, and the tests all read from the same enums.
- **Safety**: the read-only guard means the agent can never corrupt the graph, even with a hallucinated query.
- **Determinism**: same seed → identical CSV → identical graph state → reproducible agent runs (modulo Gemini sampling).
- **Future hardening**: structured Cypher tools alongside the generic `run_cypher` for high-traffic questions; vector embeddings for fuzzy product matching; multi-agent decomposition for very wide questions.

**Spec / plan trail**
- `docs/superpowers/specs/2026-05-12-synthetic-sales-data-design.md`
- `docs/superpowers/plans/2026-05-12-synthetic-sales-data.md`
- Neo4j + adk refactor plan: see `MEMORY.md` for the path